<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# Notebook 04 – Modeling: Phân lớp & Hồi quy
**Đề 2: Dự đoán Bệnh Tim**

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import sys
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, precision_recall_curve
from src.data.loader import DataLoader, load_config
from src.features.builder import FeatureBuilder
from src.models.supervised import SupervisedTrainer
from src.evaluation.metrics import Metrics
from src.evaluation.report import Reporter
from src.visualization.plots import Plotter
import shap

cfg = load_config('../configs/params.yaml')
plotter = Plotter(cfg)
reporter = Reporter(cfg)
print("Modules OK")

FileNotFoundError: [Errno 2] No such file or directory: '../configs/params.yaml'

## 1. Chuẩn bị dữ liệu

In [ ]:
df = pd.read_parquet('../data/processed/heart_processed.parquet')
builder = FeatureBuilder(cfg)
X, y = builder.get_X_y(df, extra_features=False)
X_arr = X.values
y_arr = y.values
feature_names = list(X.columns)
print(f"X: {X_arr.shape}, y: {y_arr.shape}")
print(f"Target: {np.unique(y_arr, return_counts=True)}")

In [ ]:
seed = cfg['seed']
test_size = cfg['test_size']
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, y_arr, test_size=test_size, stratify=y_arr, random_state=seed
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Cross-validation tất cả mô hình (baseline so sánh)

In [ ]:
trainer = SupervisedTrainer(cfg)
print("Cross-validation (5-fold, SMOTE):")
cv_results = trainer.train_with_cv(X_train, y_train, use_smote=True)
print("\n=== Bảng so sánh CV ===")
print(cv_results.to_string())

## 3. Train mô hình tốt nhất (XGBoost)

In [ ]:
best_name = "XGBoost"
best_model = trainer.train_final_model(X_train, y_train, model_name=best_name, use_smote=True)
eval_xgb = trainer.evaluate(best_model, X_test, y_test, model_name=best_name)

## 4. Train và đánh giá tất cả mô hình trên test set

In [ ]:
all_eval_results = []
models_roc = []
models_pr  = []
for name in ["Logistic Regression", "SVM", "Random Forest", "XGBoost"]:
    model = trainer.train_final_model(X_train, y_train, model_name=name, use_smote=True)
    ev = trainer.evaluate(model, X_test, y_test, model_name=name)
    # Metrics
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    summary = Metrics.classification_summary(y_test, y_pred, y_prob, name)
    all_eval_results.append(summary)
    # ROC/PR data
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    models_roc.append({"name": name, "fpr": fpr, "tpr": tpr, "auc": summary['ROC-AUC']})
    models_pr.append({"name": name, "precision": prec, "recall": rec, "auc": summary['PR-AUC']})

## 5. Bảng tổng hợp kết quả

In [ ]:
results_df = reporter.summarize_classification(all_eval_results)
print(results_df.to_string())

## 6. Biểu đồ so sánh

In [ ]:
plotter.plot_model_comparison(results_df)
plotter.plot_roc_curves(models_roc)
plotter.plot_pr_curves(models_pr)

## 7. Confusion Matrix & Phân tích lỗi

In [ ]:
# Mô hình tốt nhất
best_model_final = trainer.models_.get("XGBoost", best_model)
y_pred_best = best_model_final.predict(X_test)
plotter.plot_confusion_matrix(y_test, y_pred_best, "XGBoost")
print("\nConfusion Matrix:")
print(Metrics.confusion_matrix_df(y_test, y_pred_best).to_string())

In [ ]:
print("\nPhân tích lỗi (XGBoost):")
df_test = pd.DataFrame(X_test, columns=feature_names)
df_test['target'] = y_test
fn_df, fp_df = Metrics.error_analysis(y_test, y_pred_best, df_test)
print("\nFalse Negatives (bỏ sót bệnh tim):")
print(fn_df)

## 8. Feature Importance (SHAP)

In [ ]:
import shap
xgb_raw = best_model_final.named_steps['clf'] if hasattr(best_model_final, 'named_steps') else best_model_final
try:
    explainer = shap.TreeExplainer(xgb_raw)
    shap_values = explainer.shap_values(X_test)
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.savefig('../outputs/figures/19_shap_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("SHAP plot saved")
except Exception as e:
    print(f"SHAP error: {e}")
    # fallback: feature importance từ XGBoost
    import matplotlib.pyplot as plt
    fi = xgb_raw.feature_importances_
    plotter.plot_feature_importance(fi, feature_names, "XGBoost")


## 9. Hồi quy – Dự đoán huyết áp (trestbps)

In [ ]:
# Mục tiêu: dự đoán trestbps từ các đặc trưng còn lại
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split as tts
import matplotlib.pyplot as plt

# Chuẩn bị X (loại trestbps và target) và y (trestbps)
reg_cols = [c for c in feature_names if c != 'trestbps']
X_reg = df[reg_cols].values
y_reg = df['trestbps'].values
Xr_tr, Xr_te, yr_tr, yr_te = tts(X_reg, y_reg, test_size=0.2, random_state=seed)
print(f"Regression X: {Xr_tr.shape}, y range: [{y_reg.min():.0f}, {y_reg.max():.0f}]")

In [ ]:
reg_results = trainer.train_regression(Xr_tr, yr_tr, Xr_te, yr_te)
print("\n=== Kết quả Hồi quy ===")
print(reg_results.to_string())

In [ ]:
# Vẽ residual của mô hình tốt nhất (Ridge)
ridge_model = trainer.models_.get('Ridge')
if ridge_model:
    yr_pred = ridge_model.predict(Xr_te)
    plotter.plot_regression_residuals(yr_te, yr_pred, "Ridge")

## 10. Lưu mô hình tốt nhất

In [ ]:
best = trainer.models_.get("XGBoost", best_model)
trainer.save_model(best, '../outputs/models/xgboost_best.pkl')
print("Notebook 04 hoàn tất.")